### **Import Libraries**

In [1]:
import os
import pandas as pd
from google.colab import drive

from datasets import load_dataset
from google import genai

from datasets import load_dataset
from collections import defaultdict
import random
import json
import asyncio

random.seed(42)


In [2]:
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/st5230_project')


Mounted at /content/drive


### **Load Data**

In [3]:
# Load Squad dataset
print("Downloading SQuAD 2.0 dataset...")
raw_squad = load_dataset("rajpurkar/squad_v2", split="validation")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

### **0. EDA**

In [4]:
# Example values
row = raw_squad[0]
print(row['title'])
print(row['context'])
print(row['question'])
print(row['answers'])

raw_squad_df = pd.DataFrame(raw_squad)
raw_squad_df.to_excel('0_squad_raw.xlsx', index=False)


Normans
The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.
In what country is Normandy located?
{'text': ['France', 'France', 'France', 'France'], 'answer_start': [159, 159, 159, 159]}


In [5]:
# Summary: Per unique title, context
def check_same_answers(answers):
    texts = answers['text']
    if not texts:
        return None  # unanswerable - not applicable
    return len(set(texts)) == 1  # True if all identical, False if any differ

rows = []
for item in raw_squad:
    context = item['context']
    rows.append({
        'title': item['title'],
        'context': context,
        'context_length': len(context.split()),
        'is_answerable': len(item['answers']['text']) > 0,
        'answers_unanimous': check_same_answers(item['answers'])
    })
tl_cntxt = pd.DataFrame(rows)

summary_tl_cntxt = tl_cntxt.groupby(['title', 'context'])\
  .agg(
    context_length = ('context_length', 'mean'),                         # Context length -  no. of words in context
    total_qns = ('is_answerable', 'count'),                              # Total no. of qns (answerable + unanswerable)
    answerable = ('is_answerable', 'sum'),                               # No. of answerable qns (same_answers + diff_answers)
    unanswerable = ('is_answerable', lambda x: (~x).sum()),              # No. of unanswerable qns (where {'text': [], 'answer_start':[]})
    same_answers = ('answers_unanimous', lambda x: x.sum()),             # No. of answerable qns, where all given answers are the same
    diff_answers = ('answers_unanimous', lambda x: (x == False).sum())   # No. of answerable qns, where not all given answers are the same
  ).reset_index()
summary_tl_cntxt.to_excel('0_squad_eda_a.xlsx', index=False)

summary_tl_cntxt


,title,context,context_length,total_qns,answerable,unanswerable,same_answers,diff_answers
0,1973_oil_crisis,Although lacking historical connections to the...,107.0,10,5,5,2,3
1,1973_oil_crisis,An increase in imported cars into North Americ...,175.0,10,5,5,1,4
2,1973_oil_crisis,"Compact trucks were introduced, such as the To...",78.0,9,4,5,1,3
3,1973_oil_crisis,Despite being relatively unaffected by the emb...,92.0,10,5,5,1,4
4,1973_oil_crisis,"Federal safety standards, such as NHTSA Federa...",99.0,8,3,5,2,1
...,...,...,...,...,...,...,...,...
1199,Yuan_dynasty,Western medicine was also practiced in China b...,106.0,10,5,5,3,2
1200,Yuan_dynasty,Western musical instruments were introduced to...,113.0,10,5,5,3,2
1201,Yuan_dynasty,"When Yesün Temür died in Shangdu in 1328, Tugh...",202.0,10,5,5,4,1
1202,Yuan_dynasty,When the Mongols placed the Uighurs of the Kin...,88.0,8,4,4,0,4


In [6]:
# Summary: Per unqiue title
context_len_thres = 150
summary_tl_cntxt["above_thres"] = summary_tl_cntxt["context_length"] > context_len_thres
summary_tl = summary_tl_cntxt.groupby('title')\
  .agg(
      total_context = ('context', 'count'),
      below_len_thres = ('above_thres', lambda x: (~x).sum()),  # No. of context satifiying context length threshold
      above_len_thres = ('above_thres', lambda x: x.sum()),     # No. of context exceeding context length threshold
      total_qns = ('total_qns', 'sum'),
      answerable = ('answerable', 'sum'),
      unanswerable = ('unanswerable', 'sum'),
      same_answers = ('same_answers', 'sum'),
      diff_answers = ('diff_answers', 'sum')
  ).reset_index()
summary_tl_cntxt.to_excel('0_squad_eda_b.xlsx', index=False)

summary_tl_cntxt


,title,context,context_length,total_qns,answerable,unanswerable,same_answers,diff_answers,above_thres
0,1973_oil_crisis,Although lacking historical connections to the...,107.0,10,5,5,2,3,False
1,1973_oil_crisis,An increase in imported cars into North Americ...,175.0,10,5,5,1,4,True
2,1973_oil_crisis,"Compact trucks were introduced, such as the To...",78.0,9,4,5,1,3,False
3,1973_oil_crisis,Despite being relatively unaffected by the emb...,92.0,10,5,5,1,4,False
4,1973_oil_crisis,"Federal safety standards, such as NHTSA Federa...",99.0,8,3,5,2,1,False
...,...,...,...,...,...,...,...,...,...
1199,Yuan_dynasty,Western medicine was also practiced in China b...,106.0,10,5,5,3,2,False
1200,Yuan_dynasty,Western musical instruments were introduced to...,113.0,10,5,5,3,2,False
1201,Yuan_dynasty,"When Yesün Temür died in Shangdu in 1328, Tugh...",202.0,10,5,5,4,1,True
1202,Yuan_dynasty,When the Mongols placed the Uighurs of the Kin...,88.0,8,4,4,0,4,False


In [7]:
# Summary: Overall
context_len_thres = 150

# --- 'Pre' summary (before filtering) ---
pre_process = {
    'num_titles' : int(round(len(summary_tl_cntxt['title'].unique()), 0)),
    'num_context': int(len(summary_tl_cntxt['context'].unique())),
    'avg_context_len': int(round(summary_tl_cntxt['context_length'].mean(), 0)),
    'total_qns': int(summary_tl_cntxt['total_qns'].sum()),
    'answerable': int(summary_tl_cntxt['answerable'].sum()),
    'unanswerable': int(summary_tl_cntxt['unanswerable'].sum()),
    'same_answers': int(summary_tl_cntxt['same_answers'].sum()),
    'diff_answers': int(round(summary_tl_cntxt['diff_answers'].sum(), 0)) # Ensure all are int
}
pre_process_df = pd.DataFrame([pre_process])
pre_process_df['type'] = 'pre'

# --- 'Post' summary (before filtering) ---
# Re-calculate summary_tl_cntxt with the specified filtering for the 'post' summary calculation
summary_tl_cntxt = tl_cntxt[
    (tl_cntxt["context_length"] <= context_len_thres) &
    ((tl_cntxt["is_answerable"] & tl_cntxt["answers_unanimous"]) | ~tl_cntxt["is_answerable"]) # Fix: Changed 'answerable' to 'is_answerable'
  ]\
  .groupby(['title', 'context'])\
  .agg(
    context_length = ('context_length', 'mean'),                         # Context length -  no. of words in context
    total_qns = ('is_answerable', 'count'),                              # Total no. of qns (answerable + unanswerable)
    answerable = ('is_answerable', 'sum'),                               # No. of answerable qns (same_answers + diff_answers)
    unanswerable = ('is_answerable', lambda x: (~x).sum()),              # No. of unanswerable qns (where {'text': [], 'answer_start':[]})
    same_answers = ('answers_unanimous', lambda x: x.sum()),             # No. of answerable qns, where all given answers are the same
    diff_answers = ('answers_unanimous', lambda x: (x == False).sum())   # No. of answerable qns, where not all given answers are the same
  ).reset_index()

post_process = {
    'num_titles' : int(round(len(summary_tl_cntxt['title'].unique()), 0)),
    'num_context': int(len(summary_tl_cntxt['context'].unique())),
    'avg_context_len': int(round(summary_tl_cntxt['context_length'].mean(), 0)),
    'total_qns': int(summary_tl_cntxt['total_qns'].sum()),
    'answerable': int(summary_tl_cntxt['answerable'].sum()),
    'unanswerable': int(summary_tl_cntxt['unanswerable'].sum()),
    'same_answers': int(summary_tl_cntxt['same_answers'].sum()),
    'diff_answers': int(round(summary_tl_cntxt['diff_answers'].sum(), 0)) # Ensure all are int
}
post_process_df = pd.DataFrame([post_process])
post_process_df['type'] = 'post'

# Combine both
summary_overall = pd.concat([pre_process_df, post_process_df], ignore_index=True)

# Reorder columns to have 'type' first and display the combined summary
summary_overall = summary_overall[['type'] + [col for col in summary_overall.columns if col != 'type']]
summary_overall

,type,num_titles,num_context,avg_context_len,total_qns,answerable,unanswerable,same_answers,diff_answers
0,pre,35,1204,127,11873,5928,5945,2761,3167
1,post,35,924,103,6660,2101,4559,2101,0
